In [ ]:
import subprocess

RUN = "../connect/connect.ino"
# arduino_port = "/dev/ttyACM0"
arduino_port = "COM3"

def compilecode(test = True):
    file_name = RUN
    print(file_name)

    # 아두이노 코드 컴파일 명령어 실행
    compile_process = subprocess.Popen(['arduino-cli', 'compile', '--fqbn', 'arduino:avr:uno', file_name],
                                       stdout=subprocess.PIPE,
                                       stderr=subprocess.PIPE)
    
    upload_process = subprocess.Popen(['arduino-cli', 'upload', '-p', arduino_port, '--fqbn', 'arduino:avr:uno', file_name],
                                      stdout=subprocess.PIPE,
                                      stderr=subprocess.PIPE)
    
    output, error = compile_process.communicate()

    if compile_process.returncode == 0:
        return True
    else:
        print(error.decode('utf-8'))
        return False

In [38]:
compilecode()

./connect/connect.ino


True

In [43]:
import serial
import time
import asyncio
import websockets

# 아두이노와 시리얼 포트 연결 (포트 이름은 환경에 맞게 변경)
arduino = serial.Serial(port='COM3', baudrate=9600, timeout=1)

def set_led_brightness(brightness):
    if 0 <= brightness <= 255:
        arduino.write(f"{brightness}\n".encode())  # 아두이노로 데이터 전송
        print(f"LED 밝기: {brightness}")
        time.sleep(0.1)  # 전송 후 잠시 대기
    else:
        print("밝기 값은 0에서 255 사이여야 합니다.")

# WebSocket 클라이언트
async def websocket_client():
    uri = "ws://localhost:8765"  # WebSocket 서버 주소와 포트 (예: ws://localhost:8765)
    
    async with websockets.connect(uri) as websocket:
        current_brightness = -1  # 초기값
        
        while True:
            # WebSocket 서버로부터 데이터 수신
            message = await websocket.recv()
            print(f"수신된 메시지: {message}")
            
            # 수신된 메시지를 숫자로 변환
            if message.isdigit():
                brightness = int(message)
                
                # 0~255 범위 내에서만 값 전송
                if 0 <= brightness <= 255 and brightness != current_brightness:
                    set_led_brightness(brightness)
                    current_brightness = brightness
                else:
                    print("유효한 밝기 값이 아니거나 이미 설정된 값입니다.")
            else:
                print("숫자 입력만 허용됩니다.")

# 비동기 함수 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(websocket_client())


RuntimeError: This event loop is already running

In [1]:
import firebase_admin
from firebase_admin import credentials, firestore
import serial
import time

cred = credentials.Certificate("./serviceAccountKey.json")  # 서비스 계정 키 경로
firebase_admin.initialize_app(cred)

# Firebase Firestore 인스턴스
db = firestore.client()

In [6]:
doc_ref = db.collection('test').document('open')
doc = doc_ref.get()

if doc.exists:
    data = doc.to_dict()
    brightness = data.get('a', None)
brightness

15

In [7]:
data

{'a': 15, 'b': False, 'c': False}

In [ ]:
import firebase_admin
from firebase_admin import credentials, firestore
import serial
import time

# 1. Firebase 연결 설정
cred = credentials.Certificate("path/to/your/serviceAccountKey.json")  # 서비스 계정 키 경로
firebase_admin.initialize_app(cred)

# Firebase Firestore 인스턴스
db = firestore.client()

# 2. 아두이노 시리얼 포트 연결 (포트 이름은 환경에 맞게 변경)
arduino = serial.Serial(port='COM3', baudrate=9600, timeout=1)

def set_led_brightness(brightness):
    """아두이노 LED 밝기 설정"""
    if 0 <= brightness <= 255:
        arduino.write(f"{brightness}\n".encode())  # 아두이노로 데이터 전송
        print(f"LED 밝기: {brightness}")
        time.sleep(0.1)  # 전송 후 잠시 대기
    else:
        print("밝기 값은 0에서 255 사이여야 합니다.")

def get_brightness_from_firebase():
    """Firebase에서 'test' 컬렉션의 'open' 문서의 'a' 키 값을 읽어오기"""
    doc_ref = db.collection('test').document('open')
    doc = doc_ref.get()
    
    if doc.exists:
        data = doc.to_dict()
        brightness = data.get('a', None)
        
        if brightness is not None:
            print(f"Firebase에서 읽은 LED 밝기 값: {brightness}")
            return int(brightness)  # 아두이노가 사용할 수 있도록 정수로 변환
        else:
            print("Firebase 문서에 'a' 키가 없습니다.")
            return None
    else:
        print("Firebase 문서가 존재하지 않습니다.")
        return None

# 3. 메인 실행
if __name__ == "__main__":
    brightness = get_brightness_from_firebase()
    if brightness is not None:
        set_led_brightness(brightness)

    arduino.close()  # 시리얼 포트 닫기


NameError: name 'ser' is not defined